<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Machine Learning Foundations and Data Preparation — Solutions
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

*Worked answers to the **» Predict**, **» Investigate**, **» Modify** and **» Make** tasks in **04_ML_Foundations_and_Data_Preparation**, in the same order. Run the setup cell first; it rebuilds `df_stays` as the session did. The numbers quoted are for the course views. Where a task asked for a decision, the answer here is one defensible choice, not the only one.*

## Setup

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import matplotlib.pyplot as plt
import pandas_gbq

PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.chartevents, keeping itemid 220045 (heart rate)
df_hr = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.chartevents_hr`")

df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
stay_length = df_stays['dischtime'] - df_stays['admittime']
df_stays['los_days'] = stay_length.dt.total_seconds() / (60 * 60 * 24)

# The temperature columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean']:
    df_stays[column] = df_stays[column].astype(float)
df_stays['age'] = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
print(df_stays.shape)   # (9600, 54)

## Machine learning foundations

### Predict: the Datasaurus
**No.** A circle, a star, an X and a dinosaur, with the same means, standard deviations and correlation to two decimal places. The statistics are true and say nothing about the shape; the plot shows it at once. This is why EDA uses numbers and pictures together, and why `corr` close to 0 does not mean "unrelated" (the preparation notebook's point about age and length of stay).

## Exploring data for a model

### Investigate: the baseline
About **87%**. A model that always says "survives" is right for 87% of stays, so accuracy on its own cannot tell a useful model from a useless one here. Session 5 introduces the measures that can.

### Investigate: the whole-stay maximum
Median maximum heart rate over the whole stay: about **106 bpm** for survivors and **129** for those who died, against 99 and 112 for the first ICU day. The same statistic separates the groups better when it uses the whole stay.

It is a worse feature, not a better one. The whole-stay maximum includes readings from after the prediction time, and for the patients who died it includes the hours before death, when the heart rate is at its most extreme. That is why it separates so well: it has seen the outcome. A model trained with it would look good on this table and have nothing to use at 24 hours. This is **leakage**: a feature that uses information from after the prediction time. `los`, `dischtime` and `discharge_location` leak in the same way, more obviously. Section 6 counts the cost of the two windows.

In [ ]:
plausible = df_hr[(df_hr['valuenum'] >= 20) & (df_hr['valuenum'] <= 250)]
hr_max_stay = plausible.groupby('hadm_id')['valuenum'].max()
df_stays['hr_max_stay'] = df_stays['hadm_id'].map(hr_max_stay)

df_stays.groupby('died')[['heart_rate_max', 'hr_max_stay']].median()


## How does it get into the row
### Investigate: which window?
Only the **24-hour mean** could be used at 24 hours. The whole-stay mean is complete because it uses readings from the rest of the stay, including, for the patients who died, the hours before death: it leaks the outcome into the feature. The complete column uses information the model would not have at prediction time.

The ICU window is complete because the ICU stay *starts* when continuous monitoring starts: the heart rate is charted from the first minutes. The hospital window starts at `admittime`, and about 1,700 admissions reached the ICU, and so charting, later than 24 hours after that (the preparation notebook found a median of about three days). `first_day_vitalsign` is aligned to the ICU admission for exactly this reason, so for an ICU model it is the right table; for a model at hospital admission it would be leakage.

## Is it complete and correct

### Investigate: why is it missing?
| Column | Missing | Why |
| :-- | --: | :-- |
| `deathtime` | 8,396 | **Something that did not happen:** the patient did not die in hospital. The gap *is* the information. Never fill it. |
| `dod` | 5,626 | **Something that did not happen** within MIMIC's follow-up. Also information, with a different definition of death. |
| `marital_status` | 773 | **Nobody recorded it.** A category of its own (`UNKNOWN`) is honest; a mode fill invents a status. |
| `heart_rate_mean` | 10 | **Nobody measured it** in the first ICU day, which is rare enough to look at individually. |
| `temperature_mean` | 278 | **Nobody measured it**, and the next cell shows it is twice as common in stays that ended in death (5.4% against 2.5%). A median fill is defensible for the value; a `temperature_missing` column keeps what the gap says. |

Only the last two could be filled with a median "without lying", and the last one should carry an indicator.

### Investigate: error or extreme?
Fifteen stays have a first-day maximum glucose above 1,000. The values of **999,999** and **500,036** are placeholders or entry errors, not measurements. The values between **1,181 and 1,842** are extreme but possible in diabetic ketoacidosis or a hyperosmolar state, exactly the patients a model of ICU death should see.

A limit of about 2,000 mg/dL separates the two groups here; the precise number is a judgement, and the point is that it is yours, stated, not the statistic's. The z-score would be dominated by the 999,999s (as in the preparation notebook), and the IQR would flag everything above about 330 mg/dL, taking the real hyperglycaemia out with the errors.

In [ ]:
high_glucose = df_stays[df_stays['glucose_max'] > 1000]['glucose_max']
print(high_glucose.sort_values().tolist())

## Is it a number the model can work with


In [ ]:
race_counts = df_stays['race'].value_counts()
print((race_counts < 50).sum(), 'categories with fewer than 50 stays')

### Investigate: scaling with the errors in
Two admissions have a maximum heart rate above 250: **950 and 10,109 bpm**. With them in, the range runs to 10,109, and every real heart rate is squashed into its bottom fraction: median **0.006**. With them out, the median is **0.34**: a plausible heart rate sitting a third of the way up a plausible range.

In [ ]:
df_timed = df_hr.merge(df_admissions[['hadm_id', 'admittime']], on='hadm_id')
time_since_admission = df_timed['charttime'] - df_timed['admittime']
df_timed['hours'] = time_since_admission.dt.total_seconds() / (60 * 60)
df_first24h = df_timed[(df_timed['hours'] >= 0) & (df_timed['hours'] <= 24)]

hr_max_24h = df_first24h.groupby('hadm_id')['valuenum'].max()
print('Removed:', (hr_max_24h > 250).sum())
hr_max_24h = hr_max_24h[hr_max_24h <= 250]
hr_max_scaled = (hr_max_24h - hr_max_24h.min()) / (hr_max_24h.max() - hr_max_24h.min())
print('Median after scaling:', round(hr_max_scaled.median(), 3))   # about 0.34

## Independent practice

### Part 1, Modify: first stay per admission
`keep='first'` after sorting by `intime`. 8,647 rows, one per admission.

In [ ]:
df_model = df_stays.sort_values('intime').drop_duplicates('hadm_id', keep='first')
print(len(df_model))   # 8647

### Part 1, Modify: label balance
About **11.6%** of first stays end in death, against 12.6% of all stays: the repeat ICU stays within an admission are sicker patients, and counting them twice would over-represent them.

In [ ]:
df_model['died'].value_counts(normalize=True)

### Part 1, Modify: candidate vital signs
Medians by outcome, which is what the box plots show. The clearest separations are **`resp_rate_mean`** (about 18.3 against 21.0), **`sbp_min`** (92 against 83) and **`heart_rate_max`** (99 against 111); `spo2_min` (93 against 91) and `glucose_max` (155 against 171) separate less; `temperature_mean` hardly at all. Whichever of the three you compared separates most is your candidate; `heart_rate_max` is a fair one if neither of yours beat it.

In [ ]:
vitals = ['heart_rate_mean', 'heart_rate_max', 'spo2_min', 'resp_rate_mean', 'sbp_min', 'glucose_max', 'temperature_mean']
df_model.groupby('died')[vitals].median().round(1)

### Part 1, Make (stretch): test the differences
A reasonable choice is the **Mann-Whitney U test** (`scipy.stats.mannwhitneyu`), which compares two groups without assuming the values are normally distributed; the first-day vital signs have tails, so the t-test's assumption is shaky, though with this many rows it would give the same verdict. Both p-values are tiny, so both differences are real, and that is all a test says. It does not rank the features: `sbp_min` and `resp_rate_mean` are both "significant", and so would `temperature_mean` be, whose medians differ by 0.07 °C. What ranks them is the size of the separation in the box plots, and whether the column is known at prediction time. The appendix has the longer version.

In [ ]:
# Test chosen, and why: Mann-Whitney U, which does not assume a normal distribution; the vital signs have long tails
# Do the p-values tell you which is the better feature? No: both are tiny, and a p-value measures surprise, not size
from scipy.stats import mannwhitneyu

for column in ['sbp_min', 'resp_rate_mean']:
    died = df_model[df_model['died'] == 1][column].dropna()
    survived = df_model[df_model['died'] == 0][column].dropna()
    u_stat, p_value = mannwhitneyu(died, survived)
    print(column, 'p-value:', p_value)

### Part 2, Make: a basic EDA of age
Median 66, quartiles 55 and 77, minimum 18 and **maximum 99**. The 99 is the value to explain: MIMIC records every age over 89 as 91 in the anchor year, and `age` adds the years to the admission, so the oldest values are floors ("at least this old") rather than measurements. A patient recorded as 93 may be 93 or 100. There is no clean fix, which is why the age bands in the next task end with `80 and over`.

In [ ]:
# The value that needs explaining: the maximum of 99, a floor from MIMIC's cap of 91 plus the years since the anchor year
print(df_model['age'].describe().round(1))

df_model['age'].plot.hist(bins=20)
plt.xlabel('Age at admission (years)')
plt.ylabel('Stays')
plt.show()

### Part 2, Make: death by age band
The death rate rises steeply with age: about 6% under 40, 9% at 40 to 64, 14% at 65 to 79 and 17% at 80 and over.

**Number or bands?** The number keeps all the information and lets a model fit a smooth rise; the bands let it fit a different rate per band, including a non-linear one, at the cost of treating 65 and 79 as the same. For a first model, the number; the bands are worth trying if the relationship turns out not to be linear. Both is possible too, but adds columns.

In [ ]:
# Number or bands, and why: the number first, as the rate rises steadily with age; bands if a model needs the non-linearity
age_edges = [0, 40, 65, 80, 120]
age_labels = ['under 40', '40 to 64', '65 to 79', '80 and over']
df_model['age_band'] = pd.cut(df_model['age'], age_edges, labels=age_labels)

df_model.groupby('age_band', observed=True)['died'].mean().round(3)

### Part 2, Make: admission type
Grouped, the death rates are: elective about 2%, observation 12%, emergency 13%, urgent 14%. Elective stands apart; the other three are similar.

**One-hot.** The groups are not ordered: `OBSERVATION` is not "between" `ELECTIVE` and `URGENT` in any sense a model should use, and the death rates confirm it (observation and emergency are the same). Four columns is cheap. An ordinal `ELECTIVE < URGENT < EMERGENCY` would be arguable on urgency, but observation does not fit the scale.

In [ ]:
# Ordinal or one-hot, and why: one-hot, because the four groups are not ordered (observation is not between elective and urgent)
grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
df_model['admission_group'] = df_model['admission_type'].replace(grouping)
print(df_model.groupby('admission_group')['died'].mean().round(3))

adm_dummies = pd.get_dummies(df_model['admission_group'], prefix='adm')
df_model = pd.concat([df_model, adm_dummies], axis=1)
print(list(adm_dummies.columns))

### Part 2, Make: race
**13 columns.** Still to merge: the small Black subgroups (`BLACK/CARIBBEAN ISLAND`, `BLACK/CAPE VERDEAN`, `BLACK/AFRICAN`, under 70 stays each) into `BLACK/AFRICAN AMERICAN`, following the NHS-style "broad category" approach, and `PORTUGUESE` and `SOUTH AMERICAN`, which are nationalities, into `UNKNOWN/OTHER` or the nearest broad group. Keep separate: `UNKNOWN/OTHER` itself, because "not recorded" is a distinct state and over 1,300 stays, and the large groups. Whatever you merge, write it down: the model's behaviour for those patients depends on it.

In [ ]:
# Still merge: the three small Black subgroups into BLACK/AFRICAN AMERICAN; PORTUGUESE and SOUTH AMERICAN (nationalities, 48 stays) into UNKNOWN/OTHER
# Keep separate: UNKNOWN/OTHER (1,357 stays; not recorded is its own state) and every group over 300 stays
race_grouped = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_grouped = race_grouped.str.replace(' OR ', '/')
race_grouped = race_grouped.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
df_model['race_group'] = race_grouped

race_dummies = pd.get_dummies(df_model['race_group'], prefix='race')
df_model = pd.concat([df_model, race_dummies], axis=1)
print(race_dummies.shape[1], 'columns')   # 13

### Part 2, Modify: cleaning the vital signs
The largest first-day `heart_rate_max` is **277 bpm**. Supraventricular tachycardias can reach 250 or so; 277 as a sustained rate is implausible, but a single charted artefact at 277 among the maxima is exactly what `max` picks up. A limit of 250 (as in the preparation notebook) removes one value; a limit of 300 removes none. Either is defensible once stated. The lesson is that `max` is the aggregate most exposed to a single bad reading; the mean of the first day barely moves.

In [ ]:
# Heart rate limit, and why: 250 bpm, the preparation notebook's limit; above it a maximum is more likely a charting artefact than a rhythm
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

print(df_model['glucose_max'].max())
print(df_model['heart_rate_max'].max())

### Part 2, Modify: filling temperature
The fill goes into a new column, so `temperature_mean` keeps its gaps and `temperature_missing` can be made from it whenever you like; filling in place would have destroyed the gaps the first time the cell ran. The median (about 36.8 °C) is preferred to the mean because a few extreme first-day temperatures pull the mean; here they are close, so the choice matters less than the indicator does.

In [ ]:
temperature = df_model['temperature_mean']
df_model['temperature_missing'] = temperature.isna()
df_model['temperature_imputed'] = temperature.fillna(temperature.median())

print(df_model['temperature_imputed'].isna().sum())   # 0
print(df_model['temperature_missing'].sum())         # about 220 first stays

### Part 2, Make: a diabetes feature
About **30%** of first stays have a diabetes code. `isin` does the work of the merge without the merge: it asks, for each stay, whether its `hadm_id` is anywhere in the diabetes rows, so duplicate codes for one admission do not matter and the cell can be re-run safely. Checking `icd_version` as well as the prefix is the exact version (`(df_diagnoses['icd_version'] == 10) & ...`); no ICD-9 code starts with `E`, so the prefix alone is safe here. The death rate is no higher with a diabetes code than without (about 12% in both), so this feature carries little about the label; and the code is assigned after discharge, which section 5's table flags as leakage.

In [ ]:
is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
print(len(df_diabetes))

has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

print(df_model['diabetes'].mean().round(3))                    # about 0.30
print(df_model.groupby('diabetes')['died'].mean().round(3))

### Part 3, Make: assemble the table
A defensible table: the label, age, six first-day vital signs, the temperature indicator, the admission and race one-hot columns, and diabetes (kept here despite section 5's warning, so the choice is visible). The remaining gaps are the few stays with no first-day heart rate, respiratory rate or blood pressure (under 25 each), and about 210 with no usable glucose (not measured, plus the masked errors). All are dropped here, costing under 3% of the rows; the alternative for glucose is a median fill with a `glucose_missing` indicator, as for temperature. Scaling: standardisation, because the vital signs have tails and standardisation does not depend on the extremes the way min-max does. Booleans become 0/1 integers.

In [ ]:
# What I did with the remaining gaps, and why: dropped the rows (under 3%): a few stays with no heart rate, respiratory rate or blood pressure, and about 210 with no usable glucose
# Scaling, and why: standardisation of age and the vital signs, because the tails would dominate a min-max range

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max', 'temperature_imputed']
other_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)

df_features = df_model[['died'] + numeric_features + other_features].copy()
missing_counts = df_features.isna().sum()
print(missing_counts[missing_counts > 0])
df_features = df_features.dropna()

for column in numeric_features:
    values = df_features[column]
    df_features[column] = (values - values.mean()) / values.std()

for column in df_features.select_dtypes('bool').columns:
    df_features[column] = df_features[column].astype(int)

print(df_features.shape)
print(df_features.dtypes.value_counts())
df_features.describe().round(2).T[['mean', 'std', 'min', 'max']]

### Make (stretch): a MEWS heart-rate score
About 83% of stays score 0, 10% score 1, 6% score 2 and under 1% score 3. **Both** is reasonable: `heart_rate_mean` keeps the detail, and `mews_hr` gives the model the clinical thresholds directly, which a simple model cannot learn from the number alone (a linear model cannot score both 35 and 135 as "bad"). With a flexible model, the number alone may do.

In [ ]:
mews_edges = [0, 40, 50, 100, 110, 130, 1000]
mews_scores = [2, 1, 0, 1, 2, 3]
df_model['mews_hr'] = pd.cut(df_model['heart_rate_mean'], mews_edges, labels=mews_scores, ordered=False)

df_model['mews_hr'].value_counts(normalize=True).round(3)

### Make (stretch): stays shorter than a day
Removing stays under one day drops about 1,800 first stays, and the death rate barely moves (11.6% to 11.9%): the early deaths come out, but so do many more quick discharges of survivors. The point is not the rate. Those rows' outcomes were known before the prediction time, and their first-day vital signs were computed over a few hours, not a day, so they are not the same feature as a full day's summary. A study would state this exclusion in its cohort definition.

In [ ]:
df_full_day = df_model[df_model['los'] >= 1]
print(len(df_model), 'then', len(df_full_day))
print(round(df_model['died'].mean(), 3), 'then', round(df_full_day['died'].mean(), 3))

### Make (stretch): a laboratory feature
`creatinine_max` is missing for about 1% of stays, and its median is clearly higher in the stays that ended in death (1.5 against 1.0 mg/dL): a strong candidate. The few gaps can be dropped or filled with the median plus an indicator, as with temperature. `albumin_min` is missing for about 60% of stays, because albumin is not measured routinely; filling 60% of a column with a median invents more data than it keeps, so it would need the indicator approach at least, and is probably better left out of a first model.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

df_model = df_model.merge(df_labs[['stay_id', 'creatinine_max', 'albumin_min']], on='stay_id', how='left')
df_model['creatinine_max'] = df_model['creatinine_max'].astype(float)
df_model['albumin_min'] = df_model['albumin_min'].astype(float)
missing_fraction = df_model[['creatinine_max', 'albumin_min']].isna().mean()
print(missing_fraction.round(3))
print(df_model.groupby('died')['creatinine_max'].median())

## Closing: what your table depends on
1. **Let the model decide.** Twenty-four columns and a few thousand rows is what section 2 warned about: more columns make it easier for a model to fit the training table and worse at new patients, and several of the 24 are near-duplicates (min, mean and max of the same sign) that carry little new information. Letting the model choose is defensible only with the tools Session 5 brings, a held-out set to catch the overfitting, and even then a smaller table that you can explain is usually the better model. The honest version of the colleague's point is to try both and measure.
2. **Definition.** `died` is death in hospital, which is what a surgeon deciding whether to operate wants to know about this admission. `dod IS NOT NULL` counts deaths up to a year after discharge, for about a third of the survivors: a model of frailty and prognosis, not of this stay. Neither is wrong; they are different models and must not be confused.
3. **Nobody took a temperature.** The gap is a record of a clinical decision, or of a patient too unstable for routine observations, and that is what the model learns from: the hospital's behaviour, not the patient's physiology. It is a real signal in this hospital and it would vanish in one that measures everyone on arrival, so a model that leans on it does not travel. Keep it, but know what it is, and say so when the model is described.

## Appendix: significant is not the same as useful
Gender and death: p of about **0.27**, so no evidence they are related, and the death rates (13.0% for women, 12.2% for men) are close. Temperature and death: p of about 10⁻¹⁰, and the medians differ by **0.07 °C** (36.74 against 36.81).

A p-value measures how surprising the data would be if there were no relationship at all. With 9,600 rows, a tiny difference is surprising under that assumption, so it is significant, and still of no use to a model. **Significance says "not chance"; it does not say "big enough to matter".** For a feature, look at the size of the separation (the box plots) and at whether it is known at prediction time. Neither test says its feature is useful: the first finds no relationship with the label, and the second finds a real one that is too small to act on.